# Silver Layer — Transformación y Curación de Datos

**Proyecto 3 — Arquitectura Lakehouse en Azure Databricks**

Este notebook transforma los datos crudos de Bronze hacia la capa Silver con calidad y normalización.

**Características:**
- Tablas en formato **Delta**
- Escritura con **MERGE** (no override) — maneja duplicados e historia
- Validación de registros nuevos vs actualizados
- Columna `load_date` en cada tabla
- Normalización y limpieza de tipos de datos

**Tablas Silver:** silver_posts, silver_users, silver_votes, silver_comments, silver_badges, silver_postlinks

In [0]:
# ============================================================
# CELDA 1 — Parámetros de configuración Silver
# ============================================================
CATALOG        = "lacm_uao_prod_central_us"
BRONZE_SCHEMA  = "bronze"
SILVER_SCHEMA  = "silver"

BRONZE_VOLUME  = "raw_data"
SILVER_VOLUME  = "silver_data"

# Rutas via Unity Catalog Volume — sin account key
BRONZE_PATH    = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/{BRONZE_VOLUME}"
SILVER_PATH    = f"/Volumes/{CATALOG}/{SILVER_SCHEMA}/{SILVER_VOLUME}"

STORAGE_ACCOUNT = "stuaoprod001lacm"
ADLS_SILVER     = f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net"

YEAR   = 2023
MONTHS = [1, 2]

print(f"[CONFIG] Catálogo:     {CATALOG}")
print(f"[CONFIG] Bronze UC:    {CATALOG}.{BRONZE_SCHEMA}")
print(f"[CONFIG] Silver UC:    {CATALOG}.{SILVER_SCHEMA}")
print(f"[CONFIG] Bronze path:  {BRONZE_PATH}")
print(f"[CONFIG] Silver path:  {SILVER_PATH}")
print(f"[CONFIG] Período:      {MONTHS[0]:02d}/{YEAR} — {MONTHS[-1]:02d}/{YEAR}")


[CONFIG] Catálogo:     lacm_uao_prod_central_us
[CONFIG] Bronze UC:    lacm_uao_prod_central_us.bronze
[CONFIG] Silver UC:    lacm_uao_prod_central_us.silver
[CONFIG] Bronze path:  /Volumes/lacm_uao_prod_central_us/bronze/raw_data
[CONFIG] Silver path:  /Volumes/lacm_uao_prod_central_us/silver/silver_data
[CONFIG] Período:      01/2023 — 02/2023


In [0]:
# ============================================================
# CELDA 2 — Setup: imports, Spark, esquema y volumen Silver
# ⚠ Ejecutar siempre antes de las celdas de transformación
# ============================================================
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, trim, regexp_replace, when, lit,
    current_date, to_timestamp, to_date,
    year, month, length
)
from pyspark.sql.types import (
    IntegerType, LongType, StringType,
    DoubleType, BooleanType
)
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")
print(f"[OK] Esquema {CATALOG}.{SILVER_SCHEMA} listo.")

# Crear volumen Silver (managed o externo)
try:
    spark.sql(f"""
        CREATE EXTERNAL VOLUME IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}.{SILVER_VOLUME}
        LOCATION '{ADLS_SILVER}'
    """)
    print(f"[OK] Volumen externo Silver: {CATALOG}.{SILVER_SCHEMA}.{SILVER_VOLUME}")
except Exception as e:
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}.{SILVER_VOLUME}")
    print(f"[OK] Volumen managed Silver: {CATALOG}.{SILVER_SCHEMA}.{SILVER_VOLUME}")

print(f"[OK] Ruta Silver: {SILVER_PATH}")


# ── Función MERGE Silver ──────────────────────────────────────────────────────
all_metrics = {}

def merge_to_silver(df_source, table_name: str, merge_key: str, update_cols=None) -> dict:
    """
    MERGE sobre tabla Delta Silver en Unity Catalog.
    Usa spark.catalog.tableExists() — compatible con UC.
    Retorna {inserted, updated, total_source}.
    """
    table_full   = f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"
    total_source = df_source.count()

    table_exists = spark.catalog.tableExists(table_full)

    if not table_exists:
        df_source.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(table_full)
        print(f"  [CREATED] {table_full} — {total_source:,} registros")
        metrics = {"inserted": total_source, "updated": 0, "total_source": total_source}
    else:
        delta_target  = DeltaTable.forName(spark, table_full)
        existing_keys = delta_target.toDF().select(merge_key)
        matched_count = df_source.select(merge_key).join(existing_keys, merge_key, "inner").count()
        insert_count  = total_source - matched_count

        update_set = (
            {c: f"source.{c}" for c in df_source.columns if c != merge_key}
            if update_cols is None
            else {c: f"source.{c}" for c in update_cols}
        )

        delta_target.alias("target") \
            .merge(df_source.alias("source"),
                   f"target.{merge_key} = source.{merge_key}") \
            .whenMatchedUpdate(set=update_set) \
            .whenNotMatchedInsertAll() \
            .execute()

        print(f"  [MERGE] {table_full}: {insert_count:,} insertados | {matched_count:,} actualizados")
        metrics = {"inserted": insert_count, "updated": matched_count, "total_source": total_source}

    all_metrics[table_name] = metrics
    return metrics


# ── Función read_bronze — lee desde tablas Delta UC ───────────────────────────
def read_bronze(table_name: str) -> object:
    """
    Lee la tabla Bronze completa desde Unity Catalog.
    Los datos ya están registrados como Delta en UC por bronze_ingest.
    """
    full = f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
    df = spark.table(full)
    print(f"  ✓ {table_name}: {df.count():,} filas desde {full}")
    return df



# ── Función safe_num: convierte columna a número según su tipo real ──────────
def safe_num(df, col_name, target_type):
    """
    Convierte una columna a tipo numérico de forma segura,
    adaptándose al tipo actual de la columna en el DataFrame:
    - BINARY  → decode('UTF-8') → try_cast → número
    - STRING  → try_cast directo → número
    - INT/BIGINT/LONG → cast directo (ya es numérico)
    """
    from pyspark.sql.functions import expr
    col_type = dict(df.dtypes).get(col_name, 'string').lower()
    if col_type == 'binary':
        return expr(f"try_cast(decode(`{col_name}`, 'UTF-8') AS {target_type})")
    elif col_type in ('string', 'varchar'):
        return expr(f"try_cast(`{col_name}` AS {target_type})")
    else:
        # Ya es numérico (int, bigint, long, double) — cast simple
        from pyspark.sql.functions import col as _col
        if target_type.upper() in ('BIGINT', 'LONG'):
            from pyspark.sql.types import LongType as _LT
            return _col(col_name).cast(_LT())
        else:
            from pyspark.sql.types import IntegerType as _IT
            return _col(col_name).cast(_IT())

def safe_str(df, col_name):
    """
    Convierte una columna a STRING de forma segura:
    - BINARY → decode('UTF-8') → string legible
    - STRING/otros → cast a string
    """
    from pyspark.sql.functions import expr
    col_type = dict(df.dtypes).get(col_name, 'string').lower()
    if col_type == 'binary':
        return expr(f"decode(`{col_name}`, 'UTF-8')")
    else:
        from pyspark.sql.functions import col as _col
        from pyspark.sql.types import StringType as _ST
        return _col(col_name).cast(_ST())


print("[OK] Setup Silver completo — merge_to_silver y read_bronze listas.")


[OK] Esquema lacm_uao_prod_central_us.silver listo.
[OK] Volumen externo Silver: lacm_uao_prod_central_us.silver.silver_data
[OK] Ruta Silver: /Volumes/lacm_uao_prod_central_us/silver/silver_data
[OK] Setup Silver completo — merge_to_silver y read_bronze listas.


In [0]:
# ============================================================
# CELDA 3 — Silver POSTS
# safe_num() detecta el tipo real (BINARY/STRING/INT) y aplica
# la conversión correcta sin errores de tipo.
# ============================================================
print("[1/6] Transformando POSTS...")
df_posts_raw = read_bronze("posts")

df_posts_silver = df_posts_raw \
    .withColumn("Id",               safe_num(df_posts_raw, "Id",               "BIGINT")) \
    .withColumn("PostTypeId",       safe_num(df_posts_raw, "PostTypeId",       "INT")) \
    .withColumn("AcceptedAnswerId", safe_num(df_posts_raw, "AcceptedAnswerId", "BIGINT")) \
    .withColumn("Score",            safe_num(df_posts_raw, "Score",            "INT")) \
    .withColumn("ViewCount",        safe_num(df_posts_raw, "ViewCount",        "INT")) \
    .withColumn("OwnerUserId",      safe_num(df_posts_raw, "OwnerUserId",      "BIGINT")) \
    .withColumn("LastEditorUserId", safe_num(df_posts_raw, "LastEditorUserId", "BIGINT")) \
    .withColumn("AnswerCount",      safe_num(df_posts_raw, "AnswerCount",      "INT")) \
    .withColumn("CommentCount",     safe_num(df_posts_raw, "CommentCount",     "INT")) \
    .withColumn("FavoriteCount",    safe_num(df_posts_raw, "FavoriteCount",    "INT")) \
    .withColumn("ParentId",         safe_num(df_posts_raw, "ParentId",         "BIGINT")) \
    .withColumn("CreationDate",     to_timestamp(col("CreationDate"))) \
    .withColumn("LastEditDate",     to_timestamp(col("LastEditDate"))) \
    .withColumn("LastActivityDate", to_timestamp(col("LastActivityDate"))) \
    .withColumn("Title",            trim(safe_str(df_posts_raw, "Title").cast(StringType()))) \
    .withColumn("Tags",             trim(regexp_replace(safe_str(df_posts_raw, "Tags").cast(StringType()), "[<>]", " "))) \
    .withColumn("post_type_desc",
        when(col("PostTypeId") == 1, "Question")
        .when(col("PostTypeId") == 2, "Answer")
        .when(col("PostTypeId") == 5, "Wiki")
        .otherwise("Other")
    ) \
    .withColumn("is_answered",
        when(col("AcceptedAnswerId").isNotNull(), True).otherwise(False)
    ) \
    .withColumn("load_date", current_date()) \
    .dropDuplicates(["Id"])

metrics_posts = merge_to_silver(df_posts_silver, "silver_posts", merge_key="Id")
print(f"[OK] silver_posts — {metrics_posts}")


[1/6] Transformando POSTS...
  ✓ posts: 424,312 filas desde lacm_uao_prod_central_us.bronze.posts
  [CREATED] lacm_uao_prod_central_us.silver.silver_posts — 424,312 registros
[OK] silver_posts — {'inserted': 424312, 'updated': 0, 'total_source': 424312}


In [0]:
# ============================================================
# CELDA 4 — Silver USERS
# ============================================================
print("[2/6] Transformando USERS...")
df_users_raw = read_bronze("users")

df_users_silver = df_users_raw \
    .withColumn("Id",             safe_num(df_users_raw, "Id",         "BIGINT")) \
    .withColumn("Reputation",     safe_num(df_users_raw, "Reputation", "INT")) \
    .withColumn("Views",          safe_num(df_users_raw, "Views",      "INT")) \
    .withColumn("UpVotes",        safe_num(df_users_raw, "UpVotes",    "INT")) \
    .withColumn("DownVotes",      safe_num(df_users_raw, "DownVotes",  "INT")) \
    .withColumn("AccountId",      safe_num(df_users_raw, "AccountId",  "BIGINT")) \
    .withColumn("CreationDate",   to_timestamp(col("CreationDate"))) \
    .withColumn("LastAccessDate", to_timestamp(col("LastAccessDate"))) \
    .withColumn("DisplayName",    trim(safe_str(df_users_raw, "DisplayName").cast(StringType()))) \
    .withColumn("Location",
        when(col("Location").isNull(), lit("Unknown"))
        .otherwise(trim(safe_str(df_users_raw, "Location").cast(StringType())))
    ) \
    .withColumn("vote_ratio",
        when((col("UpVotes") + col("DownVotes")) > 0,
            (col("UpVotes") / (col("UpVotes") + col("DownVotes"))).cast(DoubleType())
        ).otherwise(lit(1.0))
    ) \
    .withColumn("load_date", current_date()) \
    .dropDuplicates(["Id"])

metrics_users = merge_to_silver(df_users_silver, "silver_users", merge_key="Id")
print(f"[OK] silver_users — {metrics_users}")


[2/6] Transformando USERS...
  ✓ users: 400,042 filas desde lacm_uao_prod_central_us.bronze.users
  [CREATED] lacm_uao_prod_central_us.silver.silver_users — 400,042 registros
[OK] silver_users — {'inserted': 400042, 'updated': 0, 'total_source': 400042}


In [0]:
# ============================================================
# CELDA 5 — Silver VOTES
# ============================================================
print("[3/6] Transformando VOTES...")
df_votes_raw = read_bronze("votes")

vote_type_expr = when(col("VoteTypeId") == 1, "AcceptedByOriginator") \
    .when(col("VoteTypeId") == 2, "UpMod") \
    .when(col("VoteTypeId") == 3, "DownMod") \
    .when(col("VoteTypeId") == 4, "Offensive") \
    .when(col("VoteTypeId") == 5, "Favorite") \
    .when(col("VoteTypeId") == 8, "BountyStart") \
    .when(col("VoteTypeId") == 9, "BountyClose") \
    .otherwise("Other")

df_votes_silver = df_votes_raw \
    .withColumn("Id",           safe_num(df_votes_raw, "Id",           "BIGINT")) \
    .withColumn("PostId",       safe_num(df_votes_raw, "PostId",       "BIGINT")) \
    .withColumn("VoteTypeId",   safe_num(df_votes_raw, "VoteTypeId",   "INT")) \
    .withColumn("UserId",       safe_num(df_votes_raw, "UserId",       "BIGINT")) \
    .withColumn("BountyAmount", safe_num(df_votes_raw, "BountyAmount", "INT")) \
    .withColumn("CreationDate", to_date(col("CreationDate"))) \
    .withColumn("vote_type_desc", vote_type_expr) \
    .withColumn("is_positive", when(col("VoteTypeId") == 2, True).otherwise(False)) \
    .withColumn("is_negative", when(col("VoteTypeId") == 3, True).otherwise(False)) \
    .withColumn("load_date", current_date()) \
    .dropDuplicates(["Id"])

metrics_votes = merge_to_silver(df_votes_silver, "silver_votes", merge_key="Id")
print(f"[OK] silver_votes — {metrics_votes}")


[3/6] Transformando VOTES...
  ✓ votes: 2,386,031 filas desde lacm_uao_prod_central_us.bronze.votes
  [CREATED] lacm_uao_prod_central_us.silver.silver_votes — 2,386,031 registros
[OK] silver_votes — {'inserted': 2386031, 'updated': 0, 'total_source': 2386031}


In [0]:
# ============================================================
# CELDA 6 — Silver COMMENTS
# ============================================================
print("[4/6] Transformando COMMENTS...")
df_comments_raw = read_bronze("comments")

df_comments_silver = df_comments_raw \
    .withColumn("Id",           safe_num(df_comments_raw, "Id",     "BIGINT")) \
    .withColumn("PostId",       safe_num(df_comments_raw, "PostId", "BIGINT")) \
    .withColumn("Score",        safe_num(df_comments_raw, "Score",  "INT")) \
    .withColumn("UserId",       safe_num(df_comments_raw, "UserId", "BIGINT")) \
    .withColumn("CreationDate", to_timestamp(col("CreationDate"))) \
    .withColumn("Text",         trim(safe_str(df_comments_raw, "Text").cast(StringType()))) \
    .withColumn("UserDisplayName",
        when(col("UserDisplayName").isNull(), lit("Anonymous"))
        .otherwise(trim(safe_str(df_comments_raw, "UserDisplayName").cast(StringType())))
    ) \
    .withColumn("comment_length", length(col("Text").cast(StringType())).cast(IntegerType())) \
    .withColumn("load_date", current_date()) \
    .dropDuplicates(["Id"])

metrics_comments = merge_to_silver(df_comments_silver, "silver_comments", merge_key="Id")
print(f"[OK] silver_comments — {metrics_comments}")


[4/6] Transformando COMMENTS...
  ✓ comments: 646,603 filas desde lacm_uao_prod_central_us.bronze.comments
  [CREATED] lacm_uao_prod_central_us.silver.silver_comments — 646,603 registros
[OK] silver_comments — {'inserted': 646603, 'updated': 0, 'total_source': 646603}


In [0]:
# ============================================================
# CELDA 7 — Silver BADGES
# ============================================================
print("[5/6] Transformando BADGES...")
df_badges_raw = read_bronze("badges")

df_badges_silver = df_badges_raw \
    .withColumn("Id",       safe_num(df_badges_raw, "Id",     "BIGINT")) \
    .withColumn("UserId",   safe_num(df_badges_raw, "UserId", "BIGINT")) \
    .withColumn("Class",    safe_num(df_badges_raw, "Class",  "INT")) \
    .withColumn("TagBased", col("TagBased").cast(BooleanType())) \
    .withColumn("Date",     to_timestamp(col("Date"))) \
    .withColumn("Name",     trim(safe_str(df_badges_raw, "Name").cast(StringType()))) \
    .withColumn("class_desc",
        when(col("Class") == 1, "Gold")
        .when(col("Class") == 2, "Silver")
        .when(col("Class") == 3, "Bronze")
        .otherwise("Unknown")
    ) \
    .withColumn("load_date", current_date()) \
    .dropDuplicates(["Id"])

metrics_badges = merge_to_silver(df_badges_silver, "silver_badges", merge_key="Id")
print(f"[OK] silver_badges — {metrics_badges}")


[5/6] Transformando BADGES...
  ✓ badges: 627,126 filas desde lacm_uao_prod_central_us.bronze.badges
  [CREATED] lacm_uao_prod_central_us.silver.silver_badges — 627,126 registros
[OK] silver_badges — {'inserted': 627126, 'updated': 0, 'total_source': 627126}


In [0]:
# ============================================================
# CELDA 8 — Silver POSTLINKS
# ============================================================
print("[6/6] Transformando POSTLINKS...")
df_postlinks_raw = read_bronze("postlinks")

df_postlinks_silver = df_postlinks_raw \
    .withColumn("Id",            safe_num(df_postlinks_raw, "Id",            "BIGINT")) \
    .withColumn("PostId",        safe_num(df_postlinks_raw, "PostId",        "BIGINT")) \
    .withColumn("RelatedPostId", safe_num(df_postlinks_raw, "RelatedPostId", "BIGINT")) \
    .withColumn("LinkTypeId",    safe_num(df_postlinks_raw, "LinkTypeId",    "INT")) \
    .withColumn("CreationDate",  to_timestamp(col("CreationDate"))) \
    .withColumn("link_type_desc",
        when(col("LinkTypeId") == 1, "Linked")
        .when(col("LinkTypeId") == 3, "Duplicate")
        .otherwise("Other")
    ) \
    .withColumn("load_date", current_date()) \
    .dropDuplicates(["Id"])

metrics_postlinks = merge_to_silver(df_postlinks_silver, "silver_postlinks", merge_key="Id")
print(f"[OK] silver_postlinks — {metrics_postlinks}")


[6/6] Transformando POSTLINKS...
  ✓ postlinks: 60,002 filas desde lacm_uao_prod_central_us.bronze.postlinks
  [CREATED] lacm_uao_prod_central_us.silver.silver_postlinks — 60,002 registros
[OK] silver_postlinks — {'inserted': 60002, 'updated': 0, 'total_source': 60002}


In [0]:
# ============================================================
# CELDA 9 — Reporte consolidado Silver
# ============================================================
print("\n" + "=" * 65)
print("RESUMEN DE TRANSFORMACIÓN SILVER")
print("=" * 65)
print(f"{'Tabla':<22} | {'Fuente':>8} | {'Insertados':>10} | {'Actualizados':>12}")
print("-" * 65)

total_ins = total_upd = total_src = 0
for tname, m in all_metrics.items():
    ins = m.get("inserted", 0)
    upd = m.get("updated",  0)
    src = m.get("total_source", 0)
    total_ins += ins
    total_upd += upd
    total_src += src
    print(f"{tname:<22} | {src:>8,} | {ins:>10,} | {upd:>12,}")

print("-" * 65)
print(f"{'TOTAL':<22} | {total_src:>8,} | {total_ins:>10,} | {total_upd:>12,}")
print("\n[OK] Capa Silver completada.")



RESUMEN DE TRANSFORMACIÓN SILVER
Tabla                  |   Fuente | Insertados | Actualizados
-----------------------------------------------------------------
silver_posts           |  424,312 |    424,312 |            0
silver_users           |  400,042 |    400,042 |            0
silver_votes           | 2,386,031 |  2,386,031 |            0
silver_comments        |  646,603 |    646,603 |            0
silver_badges          |  627,126 |    627,126 |            0
silver_postlinks       |   60,002 |     60,002 |            0
-----------------------------------------------------------------
TOTAL                  | 4,544,116 |  4,544,116 |            0

[OK] Capa Silver completada.


In [0]:
# ============================================================
# CELDA 10 — Validación de calidad de datos Silver
# ============================================================
print("[CALIDAD] Validando tablas Silver...\n")

silver_tables = [
    "silver_posts", "silver_users", "silver_votes",
    "silver_comments", "silver_badges", "silver_postlinks"
]

for tname in silver_tables:
    try:
        df = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.{tname}")
        total = df.count()
        nulls = df.filter(col("Id").isNull()).count()
        dups  = total - df.dropDuplicates(["Id"]).count()
        has_load = "load_date" in df.columns
        status = "✓" if nulls == 0 and dups == 0 else "⚠"
        print(f"{status} {tname:<22}: filas={total:,} | nulls_Id={nulls} | dups={dups} | load_date={has_load}")
    except Exception as e:
        print(f"✗ {tname}: {e}")


[CALIDAD] Validando tablas Silver...

✓ silver_posts          : filas=424,312 | nulls_Id=0 | dups=0 | load_date=True
✓ silver_users          : filas=400,042 | nulls_Id=0 | dups=0 | load_date=True
✓ silver_votes          : filas=2,386,031 | nulls_Id=0 | dups=0 | load_date=True
✓ silver_comments       : filas=646,603 | nulls_Id=0 | dups=0 | load_date=True
✓ silver_badges         : filas=627,126 | nulls_Id=0 | dups=0 | load_date=True
✓ silver_postlinks      : filas=60,002 | nulls_Id=0 | dups=0 | load_date=True


In [0]:
# ============================================================
# CELDA 11 — Sample de tablas Silver
# ============================================================
print("[SAMPLE] Primeras 5 filas de silver_posts:")
spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_posts") \
    .select("Id", "PostTypeId", "post_type_desc", "CreationDate", "Score", "load_date") \
    .limit(5).show(truncate=40)

print("\n[SAMPLE] Primeras 5 filas de silver_votes:")
spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_votes") \
    .select("Id", "PostId", "VoteTypeId", "vote_type_desc", "is_positive", "load_date") \
    .limit(5).show(truncate=40)

print("\n[SAMPLE] Primeras 5 filas de silver_badges:")
spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_badges") \
    .select("Id", "UserId", "Name", "class_desc", "load_date") \
    .limit(5).show(truncate=40)


[SAMPLE] Primeras 5 filas de silver_posts:
+--------+----------+--------------+-----------------------+-----+----------+
|      Id|PostTypeId|post_type_desc|           CreationDate|Score| load_date|
+--------+----------+--------------+-----------------------+-----+----------+
|74972602|         1|      Question|2023-01-01 00:07:46.847|    1|2026-05-15|
|74972610|         2|        Answer| 2023-01-01 00:11:11.39|    0|2026-05-15|
|74972614|         2|        Answer| 2023-01-01 00:13:13.95|    4|2026-05-15|
|74972628|         2|        Answer|2023-01-01 00:21:31.537|    1|2026-05-15|
|74972633|         2|        Answer|2023-01-01 00:22:39.123|    2|2026-05-15|
+--------+----------+--------------+-----------------------+-----+----------+


[SAMPLE] Primeras 5 filas de silver_votes:
+---------+--------+----------+--------------+-----------+----------+
|       Id|  PostId|VoteTypeId|vote_type_desc|is_positive| load_date|
+---------+--------+----------+--------------+-----------+----------+


In [0]:
# Instalar duckdb — compatible con Job y ejecución manual
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'duckdb', '-q'])
print('[OK] duckdb instalado.')

  Obtaining dependency information for duckdb from https://files.pythonhosted.org/packages/dc/a2/67694010693ec8c8c975e6991f48ef886d35ecbdaa2f287234882a403c21/duckdb-1.5.2-cp311-cp311-manylinux_2_26_x86_64.manylinux_2_28_x86_64.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/21.4 MB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.1/21.4 MB 1.9 MB/s eta 0:00:12
   ╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.4/21.4 MB 5.7 MB/s eta 0:00:04
   ━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.7/21.4 MB 7.4 MB/s eta 0:00:03
   ━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/21.4 MB 8.5 MB/s eta 0:00:03
   ━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/21.4 MB 9.4 MB/s eta 0:00:03
   ━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/21.4 MB 10.2 MB/s eta 0:00:02
   ━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/21.4 MB 11.1 MB/s eta 0:00:02
   ━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/21.4 MB 11.9 MB/s eta 0:00:02
   ━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/21.4 MB 12.8 MB/s eta 0

In [0]:
# ============================================================
# CELDA 12 — Consultas DuckDB sobre Silver
# ============================================================
import duckdb

con = duckdb.connect()

# --- Distribución de tipos de post ---
df_posts_pd = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_posts") \
    .select("Id", "PostTypeId", "post_type_desc", "Score", "ViewCount", "load_date") \
    .limit(5000).toPandas()

print("[DUCKDB] Distribución por tipo de post (sample 5000):")
print(con.execute("""
    SELECT post_type_desc,
           COUNT(*)        AS total_posts,
           ROUND(AVG(Score), 2) AS avg_score,
           SUM(ViewCount)  AS total_views
    FROM df_posts_pd
    GROUP BY post_type_desc
    ORDER BY total_posts DESC
""").df().to_string(index=False))

# --- Votos por tipo ---
df_votes_pd = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_votes") \
    .select("Id", "vote_type_desc", "is_positive", "is_negative") \
    .limit(10000).toPandas()

print("\n[DUCKDB] Distribución de tipos de voto (sample 10000):")
print(con.execute("""
    SELECT vote_type_desc,
           COUNT(*) AS total,
           SUM(CASE WHEN is_positive THEN 1 ELSE 0 END) AS positivos,
           SUM(CASE WHEN is_negative THEN 1 ELSE 0 END) AS negativos
    FROM df_votes_pd
    GROUP BY vote_type_desc
    ORDER BY total DESC
""").df().to_string(index=False))

print("\n[OK] Consultas DuckDB Silver completadas.")


[DUCKDB] Distribución por tipo de post (sample 5000):
post_type_desc  total_posts  avg_score  total_views
        Answer         2774       0.90          0.0
      Question         2219       0.45     943262.0
         Other            4       0.00          0.0
          Wiki            3       0.00          0.0

[DUCKDB] Distribución de tipos de voto (sample 10000):
      vote_type_desc  total  positivos  negativos
               UpMod   7212     7212.0        0.0
             DownMod   1490        0.0     1490.0
               Other    914        0.0        0.0
AcceptedByOriginator    360        0.0        0.0
         BountyStart     13        0.0        0.0
         BountyClose      9        0.0        0.0
           Offensive      2        0.0        0.0

[OK] Consultas DuckDB Silver completadas.
